In [4]:
import os

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Session

from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

In [5]:
def get_snowflake_session():
    """Initialize a Snowflake session from environment variables"""
    print("\n" + "=" * 60)
    print("🔍 Checking Environment Variables...")
    print("=" * 60)
    
    connection_parameters = {
        "account": os.getenv("SNOWFLAKE_ACCOUNT"),
        "user": os.getenv("SNOWFLAKE_USER"),
        "password": os.getenv("SNOWFLAKE_PASSWORD"),
        "role": os.getenv("SNOWFLAKE_ROLE", "AI_CORTEX_DEVELOPER_ROLE"),
        "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE", "ai_project_wh"),
        "database": os.getenv("SNOWFLAKE_DATABASE", "ai_project"),
        "schema": os.getenv("SNOWFLAKE_SCHEMA", "mlops"),
    }
  
    return Session.builder.configs(connection_parameters).create()

In [6]:
# Get the active session in Snowflake Notebooks
session = get_snowflake_session()

# Connect to Feature Store (creates if not exists)
fs = FeatureStore(
    session=session,
    database="AI_PROJECT",
    name="FEATURES",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)


🔍 Checking Environment Variables...


In [ ]:
session = get_active_session()

fs = FeatureStore(session, database="AI_PROJECT", name="FEATURES")

# Entity
hex_entity = Entity(name="HEX_ENTITY", join_keys=["HEX_ID"])
fs.register_entity(hex_entity)

# FeatureView 1: Distance features
distance_df = session.sql("""
    WITH coords AS (
        SELECT 
            t."hex_id" AS HEX_ID,
            ST_Y(H3_CELL_TO_POINT(t."hex_id")) AS LAT,
            ST_X(H3_CELL_TO_POINT(t."hex_id")) AS LON
        FROM AI_PROJECT.RAW.TRAIN t
        UNION
        SELECT 
            t."hex_id" AS HEX_ID,
            ST_Y(H3_CELL_TO_POINT(t."hex_id")) AS LAT,
            ST_X(H3_CELL_TO_POINT(t."hex_id")) AS LON
        FROM AI_PROJECT.RAW.TEST t
    ),
    distances AS (
        SELECT 
            c.HEX_ID, c.LAT, c.LON, p.CATEGORY,
            MIN(HAVERSINE(c.LAT, c.LON, p.POI_LAT, p.POI_LON)) AS MIN_DISTANCE
        FROM coords c
        CROSS JOIN AI_PROJECT.RAW.POI_COORDINATES p
        GROUP BY c.HEX_ID, c.LAT, c.LON, p.CATEGORY
    )
    SELECT 
        HEX_ID, LAT, LON,
        MAX(CASE WHEN CATEGORY = 'supermarket' THEN MIN_DISTANCE END) AS DIST_TO_SUPERMARKET,
        MAX(CASE WHEN CATEGORY = 'hospital' THEN MIN_DISTANCE END) AS DIST_TO_HOSPITAL,
        MAX(CASE WHEN CATEGORY = 'school' THEN MIN_DISTANCE END) AS DIST_TO_SCHOOL,
        MAX(CASE WHEN CATEGORY = 'park' THEN MIN_DISTANCE END) AS DIST_TO_PARK,
        MAX(CASE WHEN CATEGORY = 'restaurant' THEN MIN_DISTANCE END) AS DIST_TO_RESTAURANT,
        MAX(CASE WHEN CATEGORY = 'bank' THEN MIN_DISTANCE END) AS DIST_TO_BANK,
        MAX(CASE WHEN CATEGORY = 'cafe' THEN MIN_DISTANCE END) AS DIST_TO_CAFE,
        MAX(CASE WHEN CATEGORY = 'fuel' THEN MIN_DISTANCE END) AS DIST_TO_FUEL
    FROM distances
    GROUP BY HEX_ID, LAT, LON
""")

distance_fv = FeatureView(
    name="HEX_DISTANCE_FEATURES",
    entities=[hex_entity],
    feature_df=distance_df,
    refresh_freq=None,
    desc="Min distance from hex centroid to each POI category"
)
fs.register_feature_view(feature_view=distance_fv, version="v1")

# FeatureView 2: Hourly summary features
hourly_df = session.sql("""
    SELECT 
        "hex_id" AS HEX_ID,
        "total_hours" AS TOTAL_HOURS,
        "unique_devices" AS UNIQUE_DEVICES,
        "unique_days" AS UNIQUE_DAYS,
        "work_h1" AS WORK_H1, "work_h2" AS WORK_H2,
        "work_h3" AS WORK_H3, "work_h4" AS WORK_H4,
        "wn_h1" AS WN_H1, "wn_h2" AS WN_H2,
        "wn_h3" AS WN_H3, "wn_h4" AS WN_H4
    FROM AI_PROJECT.RAW.H8_SUMMARY_HOURS
""")

hourly_fv = FeatureView(
    name="HEX_HOURLY_FEATURES",
    entities=[hex_entity],
    feature_df=hourly_df,
    refresh_freq=None,
    desc="Hourly device activity summary per hex"
)
fs.register_feature_view(feature_view=hourly_fv, version="v1")

# FeatureView 3: POI count features
poi_df = session.sql("""
    SELECT 
        "hex_id" AS HEX_ID,
        "bank" AS BANK, "atm" AS ATM, "cafe" AS CAFE,
        "restaurant" AS RESTAURANT, "post_office" AS POST_OFFICE,
        "fuel" AS FUEL, "education" AS EDUCATION,
        "fire_station" AS FIRE_STATION, "negative" AS NEGATIVE,
        "cinema" AS CINEMA, "internet_cafe" AS INTERNET_CAFE,
        "doctors" AS DOCTORS, "car_wash" AS CAR_WASH,
        "police" AS POLICE, "supermaxi" AS SUPERMAXI,
        "bco_pichincha" AS BCO_PICHINCHA
    FROM AI_PROJECT.RAW.POINTS_OF_INTEREST_AGGREGATED
""")

poi_fv = FeatureView(
    name="HEX_POI_COUNT_FEATURES",
    entities=[hex_entity],
    feature_df=poi_df,
    refresh_freq=None,
    desc="POI counts per hex cell"
)
fs.register_feature_view(feature_view=poi_fv, version="v1")

In [ ]:
train_spine = session.sql("""
    SELECT "hex_id" AS HEX_ID, "cost_of_living" AS COST_OF_LIVING
    FROM AI_PROJECT.RAW.TRAIN
""")

train_ds = fs.generate_dataset(
    name="TRAIN_DS",
    version="v1",
    spine_df=train_spine,
    features=[distance_fv, hourly_fv, poi_fv],
    spine_label_cols=["COST_OF_LIVING"]
)